# Bayesian optimization validation notebook

This notebook validates the BAX Platform starter workflow with reusable Python code from `python/bax_platform`. It uses `temperature` and `pressure` as inputs, maximizes `yield`, and targets `median = 10`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'python').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str((PROJECT_ROOT / 'python').resolve()))
from bax_platform import InputSpec, ObjectiveSpec, make_grid, recommend_next, synthetic_experiment

CSV_PATH = PROJECT_ROOT / 'notebooks' / 'sample_bo_starting_points.csv'
INPUTS = [
    InputSpec('temperature', 200, 300, 10),
    InputSpec('pressure', 20, 60, 4),
]
OBJECTIVES = [
    ObjectiveSpec('yield', 'maximize'),
    ObjectiveSpec('median', 'target', target=10),
]
BETA = 1.2
pd.set_option('display.max_rows', 30)
pd.set_option('display.precision', 3)

In [ ]:
def make_starting_csv(path=CSV_PATH):
    starting_points = [
        (200, 20), (200, 40), (200, 60),
        (230, 28), (230, 52),
        (250, 36), (250, 44),
        (270, 28), (270, 52),
        (300, 20), (300, 40), (300, 60),
    ]
    rows = []
    for temperature, pressure in starting_points:
        yld, med = synthetic_experiment(temperature, pressure)
        rows.append({
            'temperature': temperature,
            'pressure': pressure,
            'yield': round(yld, 3),
            'median': round(med, 3),
        })
    sample = pd.DataFrame(rows)
    sample.to_csv(path, index=False)
    return sample

if not CSV_PATH.exists():
    make_starting_csv()

data = pd.read_csv(CSV_PATH)
display(data)

In [ ]:
ranked, measured, measured_front = recommend_next(data, INPUTS, OBJECTIVES, beta=BETA)
print('Recommended next experiment:')
display(ranked.head(1))
print('Top 10 candidates:')
display(ranked.head(10))
print('Measured Pareto front:')
display(measured_front)

In [ ]:
truth = make_grid(INPUTS)
truth[['yield', 'median']] = truth.apply(
    lambda row: synthetic_experiment(row['temperature'], row['pressure']),
    axis=1,
    result_type='expand',
)
yield_score = (truth['yield'] - truth['yield'].min()) / (truth['yield'].max() - truth['yield'].min())
median_score = 1 - (truth['median'] - 10).abs() / max(truth['median'].std(ddof=1), 1e-9)
truth['combined_true_utility'] = (yield_score + median_score) / 2
print('Best points on the known synthetic surface, for sanity-checking the BO recommendation:')
display(truth.sort_values('combined_true_utility', ascending=False).head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(measured['median'], measured['yield'], color='#8f9ba7', label='measured')
ax.scatter(measured_front['median'], measured_front['yield'], color='#24845d', s=80, label='measured Pareto front')
next_row = ranked.iloc[0]
ax.scatter(next_row['predicted_median'], next_row['predicted_yield'], color='#b85050', s=110, label='recommended next')
ax.axvline(10, color='#1d8ba8', linestyle='--', linewidth=1, label='median target')
ax.set_xlabel('median')
ax.set_ylabel('yield')
ax.legend()
ax.set_title('Measured front and BO recommendation')
plt.show()